# Assignment 2: Data Version Control

Setting up data versioning using DVC to track dataset shifts.

In [37]:
!pip install "dvc[gdrive]" pandas scikit-learn


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [38]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

# Init DVC and local git config
!dvc init --no-scm
!git config user.email "mohit.sinsniwal@gmail.com"
!git config user.name "Mohit Sinsniwal"

ERROR: failed to initiate DVC - '.dvc' exists. Use `-f` to force.


In [39]:
# Load and clean data
try:
    sms_df = pd.read_csv('SMSSpamCollection', sep='\t', header=None, names=['label', 'message'])
    
    # Encode labels: spam=1, ham=0
    sms_df['label'] = sms_df['label'].map({'spam': 1, 'ham': 0})
    
    print(f"Loaded {len(sms_df)} records.")
    print(sms_df.head())

    sms_df.to_csv('raw_data.csv', index=False)
    print("Saved raw_data.csv")
    
except FileNotFoundError:
    print("SMSSpamCollection not found.")

Loaded 5572 records.
   label                                            message
0      0  Go until jurong point, crazy.. Available only ...
1      0                      Ok lar... Joking wif u oni...
2      1  Free entry in 2 a wkly comp to win FA Cup fina...
3      0  U dun say so early hor... U c already then say...
4      0  Nah I don't think he goes to usf, he lives aro...
Saved raw_data.csv


In [40]:
# Create Version 1 (Seed 42)
train_v1, temp = train_test_split(sms_df, test_size=0.3, random_state=42, stratify=sms_df['label'])
val_v1, test_v1 = train_test_split(temp, test_size=0.5, random_state=42, stratify=temp['label'])

train_v1.to_csv('train.csv', index=False)
val_v1.to_csv('validation.csv', index=False)
test_v1.to_csv('test.csv', index=False)

print("Created V1 files (Seed 42)")

Created V1 files (Seed 42)


In [41]:
# Commit V1
!dvc add train.csv validation.csv test.csv
!git add train.csv.dvc validation.csv.dvc test.csv.dvc
!git add .gitignore
!git commit -m "Data V1: Seed 42"
!git tag -f v1

 ⠋ Checking graph
  0% Adding...|                          | train.csv |0/3 [00:00<?,     ?file/s]
!
                                                                                
!
  0% Checking cache in '/Users/mohit/github/AppliedMachineLearning/Assignment 2/
                                                                                
!
  0%|          |Checking out /Users/mohit/github/Appli0/1 [00:00<?,    ?files/s]
  0% Adding...|                     | validation.csv |0/3 [00:00<?,     ?file/s]
!
                                                                                
!
  0% Checking cache in '/Users/mohit/github/AppliedMachineLearning/Assignment 2/
                                                                                
!
  0%|          |Checking out /Users/mohit/github/Appli0/1 [00:00<?,    ?files/s]
  0% Adding...|                           | test.csv |0/3 [00:00<?,     ?file/s]
!
                                                                             

In [42]:
# Create Version 2 (Seed 100) - simulating data shift
train_v2, temp = train_test_split(sms_df, test_size=0.3, random_state=100, stratify=sms_df['label'])
val_v2, test_v2 = train_test_split(temp, test_size=0.5, random_state=100, stratify=temp['label'])

train_v2.to_csv('train.csv', index=False)
val_v2.to_csv('validation.csv', index=False)
test_v2.to_csv('test.csv', index=False)

print("Created V2 files (Seed 100)")

Created V2 files (Seed 100)


In [43]:
# Commit V2
!dvc add train.csv validation.csv test.csv
!git add train.csv.dvc validation.csv.dvc test.csv.dvc
!git commit -m "Data V2: Seed 100"
!git tag -f v2

 ⠋ Checking graph
  0% Adding...|                          | train.csv |0/3 [00:00<?,     ?file/s]
!
                                                                                
!
  0% Checking cache in '/Users/mohit/github/AppliedMachineLearning/Assignment 2/
                                                                                
!
  0%|          |Checking out /Users/mohit/github/Appli0/1 [00:00<?,    ?files/s]
  0% Adding...|                     | validation.csv |0/3 [00:00<?,     ?file/s]
!
                                                                                
!
  0% Checking cache in '/Users/mohit/github/AppliedMachineLearning/Assignment 2/
                                                                                
!
  0%|          |Checking out /Users/mohit/github/Appli0/1 [00:00<?,    ?files/s]
  0% Adding...|                           | test.csv |0/3 [00:00<?,     ?file/s]
!
                                                                             

In [44]:
# Quick distribution check
def check_dist(version):
    print(f"\n--- {version} Distribution ---")
    for f in ['train.csv', 'validation.csv', 'test.csv']:
        try:
            counts = pd.read_csv(f)['label'].value_counts().sort_index()
            print(f"{f}: Ham(0)={counts.get(0,0)}, Spam(1)={counts.get(1,0)}")
        except:
            pass

# Verify V1
!git checkout v1
!dvc checkout
check_dist("V1")

# Verify V2
!git checkout v2
!dvc checkout
check_dist("V2")

# Switch back to V2 for now
!git checkout v2

Previous HEAD position was ecbd9a8 Data V2: Seed 100
HEAD is now at 3d12499 Data V1: Seed 42
Building workspace index                              |3.00 [00:00,  530entry/s]
Comparing indexes                                    |4.00 [00:00, 7.78kentry/s]
Applying changes                                      |3.00 [00:00, 2.87kfile/s]
M       test.csv
M       train.csv
M       validation.csv

--- V1 Distribution ---
train.csv: Ham(0)=3377, Spam(1)=523
validation.csv: Ham(0)=724, Spam(1)=112
test.csv: Ham(0)=724, Spam(1)=112
Previous HEAD position was 3d12499 Data V1: Seed 42
HEAD is now at ecbd9a8 Data V2: Seed 100
Building workspace index                              |3.00 [00:00,  534entry/s]
Comparing indexes                                    |4.00 [00:00, 6.63kentry/s]
Applying changes                                      |3.00 [00:00, 2.98kfile/s]
M       test.csv
M       train.csv
M       validation.csv

--- V2 Distribution ---
train.csv: Ham(0)=3377, Spam(1)=523
validation.csv: 

### Remote Storage (Bonus)
Using Google Drive for remote storage.
```bash
dvc remote add -d myremote gdrive://1LPQrx-5BGe3pMhpQ-4PQSHTSX_aG0ZEp

and after that dvc push


#However in my case Google is blocks the default DVC app ("This app is blocked").
# To fix, we need to create generic Google Cloud Project and use our own credentials like this
# !dvc remote modify myremote gdrive_client_id 'YOUR_CLIENT_ID'
# !dvc remote modify myremote gdrive_client_secret 'YOUR_CLIENT_SECRET'
# and then push it again, like this
# !dvc push
# however since i cant share secrets here, so i have not tried that approach.
```

In [45]:
!dvc remote add -d myremote gdrive://1LPQrx-5BGe3pMhpQ-4PQSHTSX_aG0ZEp
!dvc push

Setting 'myremote' as a default remote.
ERROR: configuration error - config file error: remote 'myremote' already exists. Use `-f|--force` to overwrite it.
Pushing
!
  0% Checking cache in '1LPQrx-5BGe3pMhpQ-4PQSHTSX_aG0ZEp/files/md5'| |0/? [00:0/Users/mohit/anaconda3/lib/python3.11/site-packages/oauth2client/_helpers.py:255: UserWarning: Cannot access /Users/mohit/Library/Caches/pydrive2fs/710796635688-iivsgbgsb6uv1fap6635dhvuei09o66c.apps.googleusercontent.com/default.json: No such file or directory
  warnings.warn(_MISSING_FILE_MESSAGE.format(filename))
Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?client_id=710796635688-iivsgbgsb6uv1fap6635dhvuei09o66c.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive.appdata&access_type=offline&response_type=code&approval_prompt=force

^C

Pushing                                       

```#However in my case Google is blocks the default DVC app ("This app is blocked").
# To fix, we need to create generic Google Cloud Project and use our own credentials like this
# !dvc remote modify myremote gdrive_client_id 'YOUR_CLIENT_ID'
# !dvc remote modify myremote gdrive_client_secret 'YOUR_CLIENT_SECRET'
# and then push it again, like this
# !dvc push
# however since i cant share secrets here, so i have not tried that approach.
```